# UN Comtrade Pull

UN Comtrade is the United Nations database of international trade. It records how much every country buys from and sells to every other country, year by year.

**Input:** the UN Comtrade API (free tier returns 100,000 records per request, so the pull is done one year at a time).  
**Output:** `data/raw/comtrade/comtrade.csv` (total goods per country pair and year) and `data/raw/comtrade/comtrade_commodities.csv` (8 strategic HS2 chapters with shipping mode).

## Setup

First the required libraries are loaded. The API key is read from a private `.env` file. That key gives access to the Comtrade service.

In [1]:
from pathlib import Path
import os
import comtradeapicall as ct
import pandas as pd
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".projectroot").exists())
load_dotenv(ROOT / ".env")
KEY = os.environ["COMTRADE_API_KEY"]
years = ",".join(str(y) for y in range(2015, 2026))
print(f"Pulling years: {years}")

Pulling years: 2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025


## Pull the data

Because the service returns at most 100,000 records per request and a full request would be far larger, the data is pulled one year at a time and the results are stacked together. The loop prints how many records came back for each year.

In [2]:
frames = []
for y in range(2015, 2026):
    dfy = ct.getFinalData(
        subscription_key=KEY, typeCode='C', freqCode='A', clCode='HS',
        period=str(y), reporterCode=None, cmdCode='TOTAL', flowCode='X,M',
        partnerCode=None,
        partner2Code='0', customsCode='C00', motCode='0',
        format_output='JSON',
    )
    print(f'{y}: {len(dfy):,} rows')
    frames.append(dfy)
df = pd.concat(frames, ignore_index=True)
print(f'TOTAL: {len(df):,} rows')

2015: 53,033 rows
2016: 54,099 rows
2017: 54,829 rows
2018: 54,336 rows
2019: 53,742 rows
2020: 52,210 rows
2021: 53,521 rows
2022: 52,513 rows
2023: 52,021 rows
2024: 46,072 rows
2025: 27,417 rows
TOTAL: 553,793 rows


## Save

Finally everything is written to a single file inside `data/raw/`.

In [3]:
out = str(ROOT / "data/raw/comtrade/comtrade.csv")
os.makedirs(os.path.dirname(out), exist_ok=True)
df.to_csv(out, index=False)
print('Saved:', out)

Saved: /Users/oussamaennaciri/Documents/Education/Academique/Regis/MSDS692 S40 Data Science Practicum/data/raw/comtrade/comtrade.csv


Claude (Anthropic) was used only as a collaborator for writing code and polishing the notebooks. All analytical decisions, interpretations, and research were conducted independently by me.